In [0]:
# Databricks notebook source
catalog = "ecommerce_dev"

bronze_orders_table = f"{catalog}.bronze.orders"
bronze_customers_table = f"{catalog}.bronze.customers"
bronze_products_table = f"{catalog}.bronze.products"

silver_orders_table = f"{catalog}.silver.orders"
silver_customers_table = f"{catalog}.silver.customers"
silver_products_table = f"{catalog}.silver.products"

In [0]:
from pyspark.sql.functions import(
    col,
    trim,
    lower,
    initcap,
    upper,
    to_date,
    row_number
)

In [0]:
from pyspark.sql.types import(
    StringType,
    IntegerType,
    DecimalType
)

In [0]:
from pyspark.sql.window import Window

In [0]:
df_orders_bronze = spark.table(bronze_orders_table)
df_customers_bronze = spark.table(bronze_customers_table)
df_products_bronze = spark.table(bronze_products_table)

In [0]:
df_orders_bronze.printSchema()
df_customers_bronze.printSchema()
df_products_bronze.printSchema()

### To Clean Orders table

In [0]:
print("To remove extra whitespaces")
df_orders_clean = (
    df_orders_bronze
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("order_status", trim(col("order_status")))
)

In [0]:
df_orders_clean = (
    df_orders_clean
    .withColumn("order_id", col("order_id").cast("string"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("product_id", col("product_id").cast("string"))
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("quantity", col("quantity").cast("integer"))
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))
    .withColumn("order_status", col("order_status").cast("string"))
)

In [0]:
df_orders_clean = (
    df_orders_clean
    .filter(
        col("order_id").isNotNull()
        & (col("order_id") != "")
        & (col("quantity") > 0)
        & (col("unit_price") >=0)
    )
)

In [0]:
print("Groups records by `order_id` and sorts them by `_ingestion_timestamp` in descending order (newest first), so the latest record for each order can be identified.")

orders_window = Window.partitionBy("order_id").orderBy(col("_ingestion_timestamp").desc()
)

In [0]:
print("Assigns a row number to each record within its order_id group based on the latest ingestion timestamp, keeps only the most recent record (row_number = 1) for each order, and removes the temporary _row_number column to deduplicate the data.")

df_orders_clean = (
    df_orders_clean
    .withColumn("_row_number", row_number().over(orders_window))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
display(df_orders_clean)

In [0]:
print("Write the table in silver layer")

(
    df_orders_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_orders_table)
)

In [0]:
df_customers_clean = (
    df_customers_bronze
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("email", trim(col("email")))
    .withColumn("city", trim(col("city")))
    .withColumn("country", trim(col("country")))
)

In [0]:
df_customers_clean = (
    df_customers_clean
    .withColumn("email", lower(col("email")))
    .withColumn("city", initcap(col("city")))
    .withColumn("country", initcap(col("country")))
)

In [0]:
df_customers_clean = (
    df_customers_clean
    .filter(
        col("customer_id").isNotNull()
        & (col("customer_id") != "")
    )
)

In [0]:
customers_window = Window.partitionBy("customer_id").orderBy(col("_ingestion_timestamp").desc())

In [0]:
df_customers_clean = (
    df_customers_clean
    .withColumn("_row_number", row_number().over(customers_window))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
display(df_customers_clean)

In [0]:
df_customers_clean.printSchema()

In [0]:
print(
    "Duplicate customer_ids:",
    df_customers_clean.groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

In [0]:
(
    df_customers_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_customers_table)
)

In [0]:
df_products_bronze.printSchema()

In [0]:
df_products_clean = (
    df_products_bronze
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
)

In [0]:
df_products_clean = (
    df_products_clean
    .withColumn("category", upper(col("category")))
    .withColumn("category", col("category").cast("string"))
    .withColumn("product_id", col("product_id").cast("string"))
    .withColumn("product_name", col("product_name").cast("string"))
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))
)

In [0]:
df_products_clean = (
    df_products_clean
    .filter(
        col("product_id").isNotNull()
        & (col("unit_price") > 0)
    )
)

In [0]:
products_window = Window.partitionBy("product_id").orderBy(col("_ingestion_timestamp").desc())

In [0]:
display(df_products_clean)

In [0]:
df_products_clean.printSchema()

In [0]:
(
    df_products_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_products_table)
)

In [0]:
print("Create a list of tables")
silver_tables = [
    silver_orders_table,
    silver_customers_table,
    silver_products_table
]

In [0]:
for table_name in silver_tables:
    df = spark.table(table_name)

    print(f"\nTable: {table_name}")
    print(f"\nRow Count: {df.count()}")
    df.printSchema()